## Installition

In [ ]:
pip install bert_score

In [ ]:
pip install evaluate rouge_score

In [ ]:
pip install tiktoken

# Imports

In [ ]:
import pandas as pd
import torch
import time
from tqdm.auto import tqdm
import evaluate
import os




from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [5]:
benchmark_df = pd.read_csv(
    "../data/benchmark/arabic_summarization_benchmark.csv"
)

benchmark_df.head()

,text,summary
0,حلت أمس الإثنين عناصر من الفرقة الوطنية للشرطة...,اختلالات مشروع “كُورْنِيشْ آسفي”.. الفرقة الوط...
1,علمت “كود” ان ندوة الرؤساء بمجلس المستشارين ال...,نايضة على بنشماس على لفلوس عاود ثاني . علاش كن...
2,فالمغرب عارفين ومتأكدين باللي هاد القضية ديال ...,باش التقاليد كتغلب الجايحة. لمغاربة كيزورو وحد...
3,قالت مصادر محلية، أن لجنة تفتيش من وزارة الداخ...,لجنة تفتيشية من الداخلية طارت لجماعات فإقليم ا...
4,قالت صفحة “اسرائيل تتكلم بالعربية”على “الفايسب...,اسرائيل لاعبة كبير وبالزربة مع المغرب. سفيرها ...


## Qwen Model

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!


## Inference Qwen

In [8]:
def summarize(text, max_new_tokens=128):
    prompt = f"""
لخص النص التالي باللغة العربية بشكل مختصر مع الحفاظ على أهم المعلومات:

{text}
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    prediction = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return prediction.strip()

In [9]:
text = benchmark_df.iloc[0]["text"]
reference = benchmark_df.iloc[0]["summary"]

prediction = summarize(text)

print("========== MODEL ==========")
print(prediction)

print("\n========== REFERENCE ==========")
print(reference)

========== MODEL ==========
عناصر من الشرطة القضائية في آسفي اتصلوا بـ"الفرع الجهوي مراكش الجنوب" للتحقيق في قضية "كورنيش آسفي". الشكاية تشير إلى اختلالات في المشروع، قد تصل إلى جنايات. المجلس الإقليمي لآسفي يسعى لإنشاء "كورنيش"، لكن بعض السكان يشككون في المشروع.

========== REFERENCE ==========
اختلالات مشروع “كُورْنِيشْ آسفي”.. الفرقة الوطنية وصلات للمدينة وبدات الأبحاث التمهيدية


In [ ]:
results = []

for i, row in tqdm(
    benchmark_df.iterrows(),
    total=len(benchmark_df)
):
    
    start_time = time.perf_counter()
    
    prediction = summarize(row["text"])
    
    inference_time = time.perf_counter() - start_time
    
    results.append({
        "id": i,
        "text": row["text"],
        "reference_summary": row["summary"],
        "prediction": prediction,
        "inference_time": inference_time
    })

results_df = pd.DataFrame(results)

results_df.head()

In [ ]:
os.makedirs("../results", exist_ok=True)

results_df.to_csv(
    "../results/qwen2.5-1.5b-results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Results saved successfully!")

Results saved successfully!


In [14]:
qwen_results = pd.read_csv("../results/qwen2.5-1.5b-results.csv")
qwen_results.head()

,id,text,reference_summary,prediction,inference_time
0,0,حلت أمس الإثنين عناصر من الفرقة الوطنية للشرطة...,اختلالات مشروع “كُورْنِيشْ آسفي”.. الفرقة الوط...,"عناصر من الشرطة القضائية في آسفي اتصلوا بـ""الف...",4.422769
1,1,علمت “كود” ان ندوة الرؤساء بمجلس المستشارين ال...,نايضة على بنشماس على لفلوس عاود ثاني . علاش كن...,"""كود"" أفاد بأن النقاش الساخن خلال ندوة الرؤساء...",5.759217
2,2,فالمغرب عارفين ومتأكدين باللي هاد القضية ديال ...,باش التقاليد كتغلب الجايحة. لمغاربة كيزورو وحد...,ر اكيد كاين فيهم اللي كيظل يدور وفهاد الدوران ...,5.746669
3,3,قالت مصادر محلية، أن لجنة تفتيش من وزارة الداخ...,لجنة تفتيشية من الداخلية طارت لجماعات فإقليم ا...,هناك لجنة تفتيش من وزارة الداخلية في المغرب، ت...,2.361186
4,4,قالت صفحة “اسرائيل تتكلم بالعربية”على “الفايسب...,اسرائيل لاعبة كبير وبالزربة مع المغرب. سفيرها ...,"قالت صفحة ""إسرائيل تتكلم بالعربية"" على ""فيسبوك...",14.969733


## Evaluation Qwen with Rouge

In [ ]:
rouge = evaluate.load("rouge")

In [ ]:
predictions = results_df["prediction"].tolist()
references = results_df["reference_summary"].tolist()

scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False
)

scores

{'rouge1': np.float64(0.0682749138116785),
 'rouge2': np.float64(0.0015757575757575758),
 'rougeL': np.float64(0.06794911297852474),
 'rougeLsum': np.float64(0.06697454751131221)}

In [ ]:
avg_time = results_df["inference_time"].mean()
total_time = results_df["inference_time"].sum()

print(f"Average inference time: {avg_time:.2f} sec")
print(f"Total inference time: {total_time:.2f} sec")
print(f"Samples: {len(results_df)}")
print(f"Samples/minute: {60 / avg_time:.2f}")

Average inference time: 4.86 sec
Total inference time: 972.50 sec
Samples: 200
Samples/minute: 12.34


In [ ]:
results_df["prediction_words"] = (
    results_df["prediction"]
    .str.split()
    .str.len()
)

results_df["reference_words"] = (
    results_df["reference_summary"]
    .str.split()
    .str.len()
)

print(
    "Average prediction length:",
    results_df["prediction_words"].mean()
)

print(
    "Average reference length:",
    results_df["reference_words"].mean()
)

Average prediction length: 47.61
Average reference length: 14.34


## mT5 Model

In [ ]:
model_name = "yalsaffar/mt5-small-Arabic-Summarization"

tokenizer = AutoTokenizer.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
).to(device)

print("Device:", device)
print("Model loaded successfully!")

config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

c:\Users\CompuSoft\anaconda3\envs\ai_env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\CompuSoft\.cache\huggingface\hub\models--yalsaffar--mt5-small-Arabic-Summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/833 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

Device: cuda
Model loaded successfully!


## Infernece mt5

In [ ]:
text = benchmark_df.iloc[0]["text"]
reference = benchmark_df.iloc[0]["summary"]

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        no_repeat_ngram_size=2
    )

prediction = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("========== MODEL ==========")
print(prediction)

print("\n========== REFERENCE ==========")
print(reference)

========== MODEL ==========
قالت الجمعية الوطنية للشرطة القضائية المغربية لحماية المال العام إنها ستجرى من طرف الوكيل العام للملك في محكمة الإستئناف بمراكش حول وجود شبهة اختلالات في مشروع إنجاز كورنيش آسفي.

========== REFERENCE ==========
اختلالات مشروع “كُورْنِيشْ آسفي”.. الفرقة الوطنية وصلات للمدينة وبدات الأبحاث التمهيدية


In [ ]:
mt5_results = []

for i, row in tqdm(
    benchmark_df.iterrows(),
    total=len(benchmark_df)
):
    
    start_time = time.perf_counter()
    
    inputs = tokenizer(
        row["text"],
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            no_repeat_ngram_size=2
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    inference_time = time.perf_counter() - start_time

    mt5_results.append({
        "id": i,
        "text": row["text"],
        "reference_summary": row["summary"],
        "prediction": prediction,
        "inference_time": inference_time
    })

mt5_results_df = pd.DataFrame(mt5_results)

mt5_results_df.head()

  0%|          | 0/200 [00:00<?, ?it/s]

,id,text,reference_summary,prediction,inference_time
0,0,حلت أمس الإثنين عناصر من الفرقة الوطنية للشرطة...,اختلالات مشروع “كُورْنِيشْ آسفي”.. الفرقة الوط...,قالت الجمعية الوطنية للشرطة القضائية المغربية ...,1.851679
1,1,علمت “كود” ان ندوة الرؤساء بمجلس المستشارين ال...,نايضة على بنشماس على لفلوس عاود ثاني . علاش كن...,أعلن مجلس النواب البارح ان عددا من اعضاء المجل...,1.096007
2,2,فالمغرب عارفين ومتأكدين باللي هاد القضية ديال ...,باش التقاليد كتغلب الجايحة. لمغاربة كيزورو وحد...,"هناك رجل اعمال بيضاوي لـ ""كود"" في المغرب الذي ...",0.948281
3,3,قالت مصادر محلية، أن لجنة تفتيش من وزارة الداخ...,لجنة تفتيشية من الداخلية طارت لجماعات فإقليم ا...,قالت مصادر محلية وطبية لبي بي سي إن جماعة امزو...,1.582940
4,4,قالت صفحة “اسرائيل تتكلم بالعربية”على “الفايسب...,اسرائيل لاعبة كبير وبالزربة مع المغرب. سفيرها ...,وصل مكتب الاتصال الإسرائيلي في المغرب إلى الرب...,0.877870


In [ ]:
os.makedirs("../results", exist_ok=True)

mt5_results_df.to_csv(
    "../results/mt5-arabic-results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("mT5 results saved successfully!")

mT5 results saved successfully!


## Evaluation mt5 with Rouge

In [ ]:
predictions = mt5_results_df["prediction"].tolist()
references = mt5_results_df["reference_summary"].tolist()

mt5_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False
)

mt5_scores

{'rouge1': np.float64(0.027000000000000003),
 'rouge2': np.float64(0.0),
 'rougeL': np.float64(0.0275),
 'rougeLsum': np.float64(0.02691666666666667)}

In [ ]:
avg_time = mt5_results_df["inference_time"].mean()
total_time = mt5_results_df["inference_time"].sum()

mt5_results_df["prediction_words"] = (
    mt5_results_df["prediction"]
    .str.split()
    .str.len()
)

mt5_results_df["reference_words"] = (
    mt5_results_df["reference_summary"]
    .str.split()
    .str.len()
)

print(f"Average inference time: {avg_time:.2f} sec")
print(f"Total inference time: {total_time:.2f} sec")
print(f"Samples/minute: {60 / avg_time:.2f}")

print(
    "Average prediction length:",
    mt5_results_df["prediction_words"].mean()
)

print(
    "Average reference length:",
    mt5_results_df["reference_words"].mean()
)

Average inference time: 1.16 sec
Total inference time: 232.93 sec
Samples/minute: 51.52
Average prediction length: 18.11
Average reference length: 14.34


# Evaluate each Model with BERT

In [ ]:
bertscore = evaluate.load("bertscore")

In [ ]:
print("========== QWEN ==========")
print(results_df["prediction"].iloc[0])

print("\n========== mT5 ==========")
print(mt5_results_df["prediction"].iloc[0])

print("\n========== REFERENCE ==========")
print(results_df["reference_summary"].iloc[0])

========== QWEN ==========
عناصر من الشرطة القضائية في آسفي اتصلوا بـ"الفرع الجهوي مراكش الجنوب" للتحقيق في قضية "كورنيش آسفي". الشكاية تشير إلى اختلالات في المشروع، قد تصل إلى جنايات. المجلس الإقليمي لآسفي يسعى لإنشاء "كورنيش"، لكن بعض السكان يشككون في المشروع.

========== mT5 ==========
قالت الجمعية الوطنية للشرطة القضائية المغربية لحماية المال العام إنها ستجرى من طرف الوكيل العام للملك في محكمة الإستئناف بمراكش حول وجود شبهة اختلالات في مشروع إنجاز كورنيش آسفي.

========== REFERENCE ==========
اختلالات مشروع “كُورْنِيشْ آسفي”.. الفرقة الوطنية وصلات للمدينة وبدات الأبحاث التمهيدية


In [ ]:
print(
    results_df["prediction"].equals(
        mt5_results_df["prediction"]
    )
)

False


In [ ]:
qwen_predictions = results_df["prediction"].tolist()
qwen_references = results_df["reference_summary"].tolist()

qwen_bert_results = bertscore.compute(
    predictions=qwen_predictions,
    references=qwen_references,
    lang="ar"
)

qwen_bert_f1 = sum(qwen_bert_results["f1"]) / len(
    qwen_bert_results["f1"]
)

print("Qwen BERTScore F1:", qwen_bert_f1)

Qwen BERTScore F1: 0.6594252300262451


In [ ]:
mt5_predictions = mt5_results_df["prediction"].tolist()
mt5_references = mt5_results_df["reference_summary"].tolist()

mt5_bert_results = bertscore.compute(
    predictions=mt5_predictions,
    references=mt5_references,
    lang="ar"
)

mt5_bert_f1 = sum(mt5_bert_results["f1"]) / len(
    mt5_bert_results["f1"]
)

print("mT5 BERTScore F1:", mt5_bert_f1)

mT5 BERTScore F1: 0.6771834287047386
